<a href="https://colab.research.google.com/github/silvanalorens/MCC225/blob/main/XCLIP_PROJECT01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#probando el transformer
from transformers import AutoProcessor
from transformers import AutoModel

processor = AutoProcessor.from_pretrained(
    "microsoft/xclip-base-patch32"
)

model = AutoModel.from_pretrained(
    "microsoft/xclip-base-patch32"
)

print("OK")

In [ ]:
!pip install transformers
!pip install accelerate
!pip install decord
!pip install tqdm opencv-python

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# 1. Montar Google Drive (base del proyecto)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(type(model))

In [ ]:
print(type(processor))

In [ ]:
BASE_DIR = "/content/drive/MyDrive/xclip_project"

CSV_PATH = f"{BASE_DIR}/data/activitynet/activitynet_subset.csv"

VIDEO_DIR = f"{BASE_DIR}/data/activitynet/videos"

OUTPUT_DIR = f"{BASE_DIR}/outputs/embeddings"

In [ ]:

import os



EMBEDDINGS_FILE = f"{OUTPUT_DIR}/segment_embeddings.npy"

METADATA_FILE = f"{OUTPUT_DIR}/segment_metadata.csv"

PARTIAL_EMBEDDINGS_FILE = (
    f"{OUTPUT_DIR}/segment_embeddings_partial.npy"
)

PARTIAL_METADATA_FILE = (
    f"{OUTPUT_DIR}/segment_metadata_partial.csv"
)

PROGRESS_FILE = (
    f"{OUTPUT_DIR}/progress.txt"
)


files = [
    EMBEDDINGS_FILE,
    METADATA_FILE,
    PARTIAL_EMBEDDINGS_FILE,
    PARTIAL_METADATA_FILE,
    PROGRESS_FILE
]

for f in files:

    if os.path.exists(f):
        os.remove(f)
        print("Deleted:", f)

In [ ]:
# ============================================================
# 08_build_xclip_embeddings.py
#
# Genera embeddings de video usando:
# microsoft/xclip-base-patch32
#
# Dataset:
# activitynet_subset.csv
#
# Salidas:
# outputs/embeddings/
#   segment_embeddings.npy
#   segment_metadata.csv
#   progress.txt
#
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/xclip_project"

CSV_PATH = f"{BASE_DIR}/data/activitynet/activitynet_subset.csv"

VIDEO_DIR = f"{BASE_DIR}/data/activitynet/videos"

OUTPUT_DIR = f"{BASE_DIR}/outputs/embeddings"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# FILES
# ============================================================

EMBEDDINGS_FILE = f"{OUTPUT_DIR}/segment_embeddings.npy"

METADATA_FILE = f"{OUTPUT_DIR}/segment_metadata.csv"

PARTIAL_EMBEDDINGS_FILE = (
    f"{OUTPUT_DIR}/segment_embeddings_partial.npy"
)

PARTIAL_METADATA_FILE = (
    f"{OUTPUT_DIR}/segment_metadata_partial.csv"
)

PROGRESS_FILE = (
    f"{OUTPUT_DIR}/progress.txt"
)

# ============================================================
# CONFIG
# ============================================================

NUM_FRAMES = 8

SAVE_EVERY = 10

MODEL_NAME = "microsoft/xclip-base-patch32"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

# ============================================================
# MODELO
# ============================================================

print("Loading XCLIP...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)

model.eval()

print("Model loaded.")

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))

# ============================================================
# RESUME SUPPORT
# ============================================================

start_idx = 0

all_embeddings = []
metadata_rows = []

if os.path.exists(PROGRESS_FILE):

    with open(PROGRESS_FILE, "r") as f:
        start_idx = int(f.read().strip())

    print(
        f"Resuming from row {start_idx}"
    )

    if os.path.exists(PARTIAL_EMBEDDINGS_FILE):

        all_embeddings = list(
            np.load(
                PARTIAL_EMBEDDINGS_FILE
            )
        )

    if os.path.exists(PARTIAL_METADATA_FILE):

        metadata_rows = (
            pd.read_csv(
                PARTIAL_METADATA_FILE
            )
            .to_dict("records")
        )

else:

    print("Iniciando para embedding")

# ============================================================
# FRAME EXTRACTION
# ============================================================

def sample_segment_frames(
    video_path,
    start_time,
    end_time,
    num_frames=8
):
    """
    Extrae exactamente num_frames frames.

    """

    cap = cv2.VideoCapture(str(video_path))

    timestamps = np.linspace(
        start_time,
        end_time,
        num_frames
    )

    frames = []

    for t in timestamps:

        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            t * 1000
        )

        ok, frame = cap.read()

        if ok:

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frames.append(frame)

    cap.release()

    # si no obtuvo ningún frame
    if len(frames) == 0:
        return None

    # completar hasta tener exactamente 8
    while len(frames) < num_frames:

        frames.append(
            frames[-1].copy()
        )

    # por seguridad
    frames = frames[:num_frames]

    return frames
# ============================================================
# MAIN LOOP
# ============================================================

for idx in tqdm(
    range(start_idx, len(df))
    #range(0,10)
):

    row = df.iloc[idx]

    video_path = os.path.join(
        VIDEO_DIR,
        row["video_file"]
    )

    if not os.path.exists(video_path):

        print(
            f"Missing video: {video_path}"
        )
        continue

    frames = sample_segment_frames(
        video_path,
        row["start"],
        row["end"],
        NUM_FRAMES
    )

    if frames is None:

        print(
            f"Failed frame extraction: {video_path}"
        )
        continue

    try:

        inputs = processor(
            images=frames,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = model.get_video_features(
                pixel_values=inputs["pixel_values"]
            )

        # Compatible con distintas versiones
        if hasattr(outputs, "pooler_output"):
            embedding = outputs.pooler_output
        else:
            embedding = outputs

        embedding = torch.nn.functional.normalize(
            embedding,
            p=2,
            dim=-1
        )

        embedding = (
            embedding
            .cpu()
            .numpy()
        )

        all_embeddings.append(
            embedding
        )

        metadata_rows.append({
            "video_id":
                row["video_id"],

            "youtube_id":
                row["youtube_id"],

            "video_file":
                row["video_file"],

            "segment_id":
                row["segment_id"],

            "start":
                row["start"],

            "end":
                row["end"],

            "caption":
                row["caption"]
        })

    except Exception as e:

        print(
            f"Error row {idx}: {e}"
        )

        continue

    # ========================================================
    # CHECKPOINT
    # ========================================================

    processed = len(
        all_embeddings
    )

    if (
        processed > 0
        and
        processed % SAVE_EVERY == 0
    ):

        np.save(
            PARTIAL_EMBEDDINGS_FILE,
            np.vstack(
                all_embeddings
            )
        )

        pd.DataFrame(
            metadata_rows
        ).to_csv(
            PARTIAL_METADATA_FILE,
            index=False
        )

        with open(
            PROGRESS_FILE,
            "w"
        ) as f:

            f.write(
                str(idx + 1)
            )

        print(
            f"Checkpoint saved "
            f"({processed} segments)"
        )

# ============================================================
# FINAL SAVE
# ============================================================

print("Guardando archivos...")

embeddings = np.vstack(
    all_embeddings
)

metadata = pd.DataFrame(
    metadata_rows
)

np.save(
    EMBEDDINGS_FILE,
    embeddings
)

metadata.to_csv(
    METADATA_FILE,
    index=False
)

with open(
    PROGRESS_FILE,
    "w"
) as f:

    f.write(
        str(len(df))
    )

print()
print("Hecho.")
print(
    "tamaño de Embeddings",
    embeddings.shape
)
print(
    "Saved:",
    EMBEDDINGS_FILE
)
print(
    "Saved:",
    METADATA_FILE
)

In [ ]:
#prueba antes del embedding texto
caption = "A man is playing piano"

inputs = processor(
    text=[caption],
    return_tensors="pt",
    padding=True,
    truncation=True
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.get_text_features(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

print(type(outputs))

if hasattr(outputs, "pooler_output"):
    print(outputs.pooler_output.shape)
else:
    print(outputs.shape)

In [ ]:
# ============================================================
# 09_build_text_embeddings.py
#
# Genera embeddings de texto usando X-CLIP
#
# Salidas:
#   text_embeddings.npy
#   text_metadata.csv
#
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

# ============================================================
# RUTAS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/xclip_project"

CSV_PATH = (
    f"{BASE_DIR}/data/activitynet/activitynet_subset.csv"
)

OUTPUT_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

TEXT_EMBEDDINGS_FILE = (
    f"{OUTPUT_DIR}/text_embeddings.npy"
)

TEXT_METADATA_FILE = (
    f"{OUTPUT_DIR}/text_metadata.csv"
)

# ============================================================
# CONFIGURACION
# ============================================================

MODEL_NAME = (
    "microsoft/xclip-base-patch32"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

# ============================================================
# MODELO
# ============================================================

print("Loading XCLIP...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)

model.eval()

print("Model loaded.")

# ============================================================
# DATA
# ============================================================

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))

# ============================================================
# BUILD TEXT EMBEDDINGS
# ============================================================

all_embeddings = []

for caption in tqdm(df["caption"]):

    inputs = processor(
        text=[str(caption)],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.get_text_features(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

    if hasattr(outputs, "pooler_output"):
        embedding = outputs.pooler_output
    else:
        embedding = outputs

    embedding = torch.nn.functional.normalize(
        embedding,
        p=2,
        dim=-1
    )

    embedding = embedding.squeeze(0)

    embedding = embedding.cpu().numpy()

    all_embeddings.append(
        embedding
    )

# ============================================================
# GUARDAR
# ============================================================

embeddings = np.vstack(
    all_embeddings
)

np.save(
    TEXT_EMBEDDINGS_FILE,
    embeddings
)

df.to_csv(
    TEXT_METADATA_FILE,
    index=False
)

print()
print("Done.")
print(
    "Embeddings shape:",
    embeddings.shape
)

print(
    "Saved:",
    TEXT_EMBEDDINGS_FILE
)

In [ ]:
import numpy as np

video = np.load(
    "/content/drive/MyDrive/xclip_project/outputs/embeddings/segment_embeddings.npy"
)

text = np.load(
    "/content/drive/MyDrive/xclip_project/outputs/embeddings/text_embeddings.npy"
)

print("Video:", video.shape)
print("Text :", text.shape)

print("Video norm:", np.linalg.norm(video[0]))
print("Text norm :", np.linalg.norm(text[0]))

In [ ]:
#10_text_to_video_retrieval.py
import numpy as np
import pandas as pd
import torch

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/xclip_project"

OUTPUT_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

VIDEO_EMB_FILE = (
    f"{OUTPUT_DIR}/segment_embeddings.npy"
)

TEXT_EMB_FILE = (
    f"{OUTPUT_DIR}/text_embeddings.npy"
)

META_FILE = (
    f"{OUTPUT_DIR}/segment_metadata.csv"
)

# ============================================================
# LOAD
# ============================================================

video_embeddings = np.load(
    VIDEO_EMB_FILE
)

text_embeddings = np.load(
    TEXT_EMB_FILE
)

metadata = pd.read_csv(
    META_FILE
)

print("Video:", video_embeddings.shape)
print("Text :", text_embeddings.shape)

# ============================================================
# QUERY
# ============================================================

query_idx = 0

query_caption = metadata.iloc[
    query_idx
]["caption"]
print(metadata.iloc[0])

query_embedding = text_embeddings[
    query_idx
]

print()
print("QUERY:")
print(query_caption)

# ============================================================
# SIMILARIDAD
# ============================================================

scores = (
    video_embeddings
    @
    query_embedding
)

# ============================================================
# TOP K
# ============================================================

TOP_K = 5

top_indices = np.argsort(
    scores
)[::-1][:TOP_K]

print()
print("TOP RESULTS")
print("-" * 50)

for rank, idx in enumerate(
    top_indices,
    start=1
):

    row = metadata.iloc[idx]

    print()
    print(f"Rank {rank}")
    print(
        f"Score: {scores[idx]:.4f}"
    )
    print(
        f"Video: {row['video_file']}"
    )
    print(
        f"Caption: {row['caption']}"
    )

In [ ]:
#11 construccion del indice FAISS
!pip install -q faiss-cpu

In [ ]:
# 11_build_faiss_index.py

import os
import numpy as np
import faiss

BASE_DIR = "/content/drive/MyDrive/xclip_project"

OUTPUT_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

INDEX_DIR = (
    f"{BASE_DIR}/outputs/faiss"
)

os.makedirs(
    INDEX_DIR,
    exist_ok=True
)

VIDEO_EMB_FILE = (
    f"{OUTPUT_DIR}/segment_embeddings.npy"
)

INDEX_FILE = (
    f"{INDEX_DIR}/video_index.faiss"
)

# =====================================================
# LOAD EMBEDDINGS
# =====================================================

embeddings = np.load(
    VIDEO_EMB_FILE
)

embeddings = embeddings.astype(
    np.float32
)

print(
    "Embeddings:",
    embeddings.shape
)

# =====================================================
# BUILD INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

print(
    "Vectors indexed:",
    index.ntotal
)

# =====================================================
# GUARDAR
# =====================================================

faiss.write_index(
    index,
    INDEX_FILE
)

print()
print("Saved:")
print(INDEX_FILE)

In [ ]:
#11_build_faiss_text_index.py
import os
import numpy as np
import faiss

BASE_DIR = "/content/drive/MyDrive/xclip_project"

OUTPUT_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

INDEX_DIR = (
    f"{BASE_DIR}/outputs/faiss"
)

os.makedirs(
    INDEX_DIR,
    exist_ok=True
)

TEXT_EMB_FILE = (
    f"{OUTPUT_DIR}/text_embeddings.npy"
)

INDEX_FILE = (
    f"{INDEX_DIR}/text_index.faiss"
)

# ==========================================
# LOAD
# ==========================================

embeddings = np.load(
    TEXT_EMB_FILE
).astype(np.float32)

print(
    "Embeddings:",
    embeddings.shape
)

# ==========================================
# BUILD INDEX
# ==========================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

print(
    "Indexed:",
    index.ntotal
)

# ==========================================
# GUARDAR
# ==========================================

faiss.write_index(
    index,
    INDEX_FILE
)

print()
print("Saved:")
print(INDEX_FILE)

In [ ]:
#12 video to text retrieval
import numpy as np
import pandas as pd
import faiss

BASE_DIR = "/content/drive/MyDrive/xclip_project"

EMB_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

INDEX_DIR = (
    f"{BASE_DIR}/outputs/faiss"
)

video_embeddings = np.load(
    f"{EMB_DIR}/segment_embeddings.npy"
).astype(np.float32)

metadata = pd.read_csv(
    f"{EMB_DIR}/segment_metadata.csv"
)

index = faiss.read_index(
    f"{INDEX_DIR}/text_index.faiss"
)

# ==========================================
# QUERY VIDEO
# ==========================================

query_idx = 0

query_video = video_embeddings[
    query_idx
]

query_video = query_video.reshape(
    1,
    -1
)

TOP_K = 5

scores, indices = index.search(
    query_video,
    TOP_K
)

print()
print("VIDEO:")
print(
    metadata.iloc[query_idx]["video_file"]
)

print()
print("TOP CAPTIONS")
print("-" * 50)

for rank, idx in enumerate(
    indices[0],
    start=1
):

    print()
    print("Rank", rank)
    print(
        "Score:",
        float(scores[0][rank-1])
    )
    print(
        metadata.iloc[idx]["caption"]
    )

In [ ]:
# Script 13 Métricas Recall@K
import numpy as np

# ============================================================
# LOAD
# ============================================================

BASE_DIR = "/content/drive/MyDrive/xclip_project"

OUTPUT_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

video_embeddings = np.load(
    f"{OUTPUT_DIR}/segment_embeddings.npy"
)

text_embeddings = np.load(
    f"{OUTPUT_DIR}/text_embeddings.npy"
)

N = len(video_embeddings)

print("Samples:", N)

# ============================================================
# TEXT -> VIDEO
# ============================================================

r1 = 0
r5 = 0
r10 = 0

for i in range(N):

    scores = (
        video_embeddings
        @
        text_embeddings[i]
    )

    ranking = np.argsort(
        scores
    )[::-1]

    rank = np.where(
        ranking == i
    )[0][0] + 1

    if rank <= 1:
        r1 += 1

    if rank <= 5:
        r5 += 1

    if rank <= 10:
        r10 += 1

print()
print("TEXT -> VIDEO")

print(
    "Recall@1 :",
    r1 / N
)

print(
    "Recall@5 :",
    r5 / N
)

print(
    "Recall@10:",
    r10 / N
)

# ============================================================
# VIDEO -> TEXT
# ============================================================

r1 = 0
r5 = 0
r10 = 0

for i in range(N):

    scores = (
        text_embeddings
        @
        video_embeddings[i]
    )

    ranking = np.argsort(
        scores
    )[::-1]

    rank = np.where(
        ranking == i
    )[0][0] + 1

    if rank <= 1:
        r1 += 1

    if rank <= 5:
        r5 += 1

    if rank <= 10:
        r10 += 1

print()
print("VIDEO -> TEXT")

print(
    "Recall@1 :",
    r1 / N
)

print(
    "Recall@5 :",
    r5 / N
)

print(
    "Recall@10:",
    r10 / N
)

In [ ]:
#14 plots.py
# ============================================================
# 14_plots.py
#
# Graficos para X-CLIP Retrieval
#
# Salidas:
# outputs/figures/
#   recall_at_k.png
#   rank_distribution.png
#   similarity_matrix.png
#
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/xclip_project"

EMB_DIR = (
    f"{BASE_DIR}/outputs/embeddings"
)

FIG_DIR = (
    f"{BASE_DIR}/outputs/figures"
)

os.makedirs(
    FIG_DIR,
    exist_ok=True
)

VIDEO_EMB_FILE = (
    f"{EMB_DIR}/segment_embeddings.npy"
)

TEXT_EMB_FILE = (
    f"{EMB_DIR}/text_embeddings.npy"
)

# ============================================================
# LOAD
# ============================================================

video_embeddings = np.load(
    VIDEO_EMB_FILE
)

text_embeddings = np.load(
    TEXT_EMB_FILE
)

print("Video:", video_embeddings.shape)
print("Text :", text_embeddings.shape)

# ============================================================
# SIMILARIDAD MATRIX
# ============================================================

similarity = (
    text_embeddings
    @
    video_embeddings.T
)

print(
    "Similarity:",
    similarity.shape
)

# ============================================================
# RECALL METRICS
# ============================================================

N = similarity.shape[0]

r1 = 0
r5 = 0
r10 = 0

ranks = []

for i in range(N):

    ranking = np.argsort(
        similarity[i]
    )[::-1]

    rank = (
        np.where(
            ranking == i
        )[0][0]
        + 1
    )

    ranks.append(rank)

    if rank <= 1:
        r1 += 1

    if rank <= 5:
        r5 += 1

    if rank <= 10:
        r10 += 1

r1 /= N
r5 /= N
r10 /= N

print()
print("Recall@1 :", r1)
print("Recall@5 :", r5)
print("Recall@10:", r10)

# ============================================================
# PLOT 1
# ============================================================

plt.figure(figsize=(6,4))

plt.bar(
    ["R@1","R@5","R@10"],
    [r1,r5,r10]
)

plt.ylim(0,1)

plt.ylabel("Recall")

plt.title(
    "X-CLIP Text → Video Retrieval"
)

plt.tight_layout()

plt.savefig(
    f"{FIG_DIR}/recall_at_k.png"
)

plt.close()

# ============================================================
# PLOT 2
# ============================================================

plt.figure(figsize=(7,4))

plt.hist(
    ranks,
    bins=20
)

plt.xlabel(
    "Ranking del video correcto"
)

plt.ylabel(
    "Cantidad de consultas"
)

plt.title(
    "Distribucion de Rankings"
)

plt.tight_layout()

plt.savefig(
    f"{FIG_DIR}/rank_distribution.png"
)

plt.close()

# ============================================================
# PLOT 3
# ============================================================

plt.figure(figsize=(8,6))

plt.imshow(
    similarity,
    aspect="auto"
)

plt.colorbar()

plt.xlabel(
    "Videos"
)

plt.ylabel(
    "Consultas de texto"
)

plt.title(
    "Matriz de Similitud Texto-Video"
)

plt.tight_layout()

plt.savefig(
    f"{FIG_DIR}/similarity_matrix.png"
)

plt.close()

# ============================================================
# DONE
# ============================================================

print()
print("Figures saved:")
print()

print(
    f"{FIG_DIR}/recall_at_k.png"
)

print(
    f"{FIG_DIR}/rank_distribution.png"
)

print(
    f"{FIG_DIR}/similarity_matrix.png"
)